[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Series


## What you will be able to do

Build a Series from a list, a dictionary or a column of a table, read a value by its label or by its
position, and predict what arithmetic between two Series does when their labels differ.

## The idea

### The problem

The **Why Pandas** notebook asked a table of 35,040 readings for each station's mean and got back four
numbers with the station names attached. That answer was a Series, and so is every column of the
table. Most of what pandas does happens to a Series, so its rules decide what your answers mean.

Some of those rules surprise people who arrive from lists and arrays. Every value carries a label,
including values that never had one. A Series made from a list numbers its values 0, 1, 2 and so on,
and a column filtered down to Oslo's readings keeps the labels those rows had in the full table, 1, 5,
9 and onward. The labels decide what arithmetic does, too. Subtract one Series from another and pandas
pairs the values by label before it subtracts, so two Series with no labels in common give a result
that is missing everywhere, with no error.

Positions and labels also share the square brackets. `s[0]` can mean the value labeled 0 or the first
value, and the answer depends on what kind of labels the Series has. pandas 2.2 treats `s[0]` as a position
when the labels are text, and warns that a future version will treat it as a label.

### What a Series is

> A **Series** is a one-dimensional array of values of one type, together with an **index**, a label for
> every value, and an optional **name**. Arithmetic between two Series matches their values by label, and
> a label that only one side has gives a missing value, `NaN`.

### Why it works that way

The index is part of the Series, the way a column's name is part of a spreadsheet. A value keeps its
label through every operation that keeps the value, so a Series filtered, sorted or converted to
Fahrenheit still says which station or which hour each number belongs to.

Matching by label is what makes results from different places safe to combine. Each station's highest
reading and each station's lowest can come from two separate calculations, in two different orders, and
subtracting one from the other still pairs Bergen with Bergen. When one side has a label the other lacks,
pandas cannot invent a value for it, so the result holds `NaN` there, and the labels of the result are
every label from either side.

Reading one value has two forms, which the **loc and iloc** notebook takes further. `s.loc[...]` looks up
a label, and `s.iloc[...]` counts positions from 0. Plain square brackets choose for you. With text labels
`s["Oslo"]` is a label and `s[0]` is a position, with a warning in pandas 2.2.

### Where this shows up

Every column you take from a DataFrame is a Series, and so is the answer from `groupby` with one column
and one statistic, such as `mean`, from `value_counts`, and from `describe` on a single column.
**The Index** notebook makes the index into something chosen on purpose, such as the time of each reading,
and the **Merging** notebook lines up whole tables by a key the way a Series lines up by label.

### What this notebook covers

- a Series from a list, from a dictionary, and from a column of the table
- reading a value by label with `loc` and by position with `iloc`
- arithmetic and comparisons on every value, with the labels kept
- the labels a filtered column keeps, and why they are not 0, 1, 2
- two Series lined up by label, including a label only one of them has, and a comparison that needs the
  labels lined up first
- a Series indexed by time, so that two stations line up hour by hour

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import pandas as pd

means = pd.Series({"Bergen": 8.0, "Oslo": 6.5})
highs = pd.Series({"Oslo": 19.3, "Bergen": 20.8, "Tromso": 16.2})
print(highs - means)
```

```
Bergen    12.8
Oslo      12.8
Tromso     NaN
dtype: float64
```

`highs` lists Oslo first and `means` lists Bergen first, and the subtraction paired them by name anyway.
Tromso is in `highs` and not in `means`, so its result is `NaN`, and the answer's labels are the stations
from both sides, sorted.

## Setup

One library, the helpers that build the readings, and one import for a warning.

- `pandas`, imported as `pd`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings
- `warnings` catches the warning pandas gives in Common errors, so the cell can print its first line

`readings` is the table of 35,040 hourly readings at the four stations in `STATIONS`, made by the same
formula as the **sqlite3, Deep Dive** guide, with Svalbard silent on 2 March. `stations` holds each
station's latitude, with Kirkenes, which has no readings. Both are the tables the **Why Pandas** notebook
built.

In [1]:
import math
import warnings
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


## Worked examples

### A Series from a list

A Series made from a list gets labels nobody chose, the positions from 0:

In [2]:
first_hours = pd.Series([-3.6, -3.7, -3.7, -3.8])

print(first_hours)
print(first_hours.index)

0   -3.6
1   -3.7
2   -3.7
3   -3.8
dtype: float64
RangeIndex(start=0, stop=4, step=1)


`RangeIndex(start=0, stop=4, step=1)` is pandas' compact way of storing the labels 0 to 3. Under the
values, `dtype: float64` gives their type, which is one type for the whole Series, as in a NumPy array.

### A Series from a dictionary

A dictionary already pairs labels with values, so its keys become the index. `STATIONS` holds the mean
each station's formula is built around:

In [3]:
planned = pd.Series(STATIONS)

print(planned)
print(planned.index)
print("name:", planned.name)

Bergen      8.0
Oslo        6.5
Svalbard   -4.5
Tromso      3.5
dtype: float64
Index(['Bergen', 'Oslo', 'Svalbard', 'Tromso'], dtype='object')
name: None


The index is the four station names, kept in the dictionary's order. The Series has no name, because
nothing gave it one. A column taken from a table is named after the column.

### Reading one value by label and by position

`loc` looks up a label, and `iloc` counts positions from 0. Oslo is the label `"Oslo"` and the value at
position 1:

In [4]:
print(planned.loc["Oslo"])
print(planned.iloc[1])
print(planned["Oslo"])

6.5
6.5
6.5


All three give 6.5. Square brackets with a text label look the label up, which is the common way to
write it when the labels are text. The **loc and iloc** notebook covers slices and lists of labels.

### Arithmetic on every value keeps the labels

In [5]:
print(planned * 9 / 5 + 32)
print(planned > 0)
print(planned[planned > 0])

Bergen      46.4
Oslo        43.7
Svalbard    23.9
Tromso      38.3
dtype: float64
Bergen       True
Oslo         True
Svalbard    False
Tromso       True
dtype: bool
Bergen    8.0
Oslo      6.5
Tromso    3.5
dtype: float64


The conversion to Fahrenheit applied to every value and kept every label. `planned > 0` is a Series of
`True` and `False` with the same labels, and putting it in square brackets kept the three stations whose
mean is above freezing and left out Svalbard.

### A column of the table is a Series

`readings["celsius"]` is a Series named `celsius`, with the table's index:

In [6]:
celsius = readings["celsius"]

print(celsius.name, celsius.dtype, len(celsius))
print(celsius.index)
print(celsius.to_numpy()[:4])

celsius float64 35040
RangeIndex(start=0, stop=35040, step=1)
[ -3.6  -3.5 -14.6  -6.7]


The index is the table's row labels, 0 to 35039. `to_numpy()` returns the values without their labels,
as a NumPy array, the type the **NumPy, Deep Dive** guide worked with.

### A filtered column keeps its labels

Oslo's readings are every fourth row of the table, starting at row 1. Filtering keeps the rows and their
labels:

In [7]:
oslo = readings[readings["station"] == "Oslo"]["celsius"]

print(oslo.head(3))
print(oslo.index[:5].tolist(), "...", oslo.index[-1])

1   -3.5
5   -5.3
9   -5.3
Name: celsius, dtype: float64
[1, 5, 9, 13, 17] ... 35037


The labels are 1, 5, 9, 13 and onward, ending at 35037, which are the rows Oslo had in `readings`. This
is the index you did not ask for. It is still useful, since label 5 points back to row 5 of the table,
and it means `loc` and `iloc` now disagree about small numbers:

In [8]:
print("label 5:", oslo.loc[5])
print("position 1:", oslo.iloc[1])
print("position 5:", oslo.iloc[5])
print("label of position 5:", oslo.index[5])

label 5: -5.3
position 1: -5.3
position 5: -4.1
label of position 5: 21


Label 5 and position 1 are the same reading, Oslo at 01:00 on 1 January. Position 5 is Oslo's sixth
reading, at 05:00, and lives at label 21.

### Lining up Series by label

Each station's highest reading comes from one calculation, and each station's lowest from another, sorted
from coldest to warmest so that the two are in different orders:

In [9]:
highest = readings.groupby("station")["celsius"].max()
lowest = readings.groupby("station")["celsius"].min().sort_values()

print(highest)
print(lowest)
print(highest - lowest)

station
Bergen      20.8
Oslo        19.3
Svalbard     8.3
Tromso      16.2
Name: celsius, dtype: float64
station
Svalbard   -17.3
Tromso      -9.2
Oslo        -6.3
Bergen      -4.8
Name: celsius, dtype: float64
station
Bergen      25.6
Oslo        25.6
Svalbard    25.6
Tromso      25.4
Name: celsius, dtype: float64


`lowest` runs Svalbard, Tromso, Oslo, Bergen, and `highest` runs in alphabetical order. The subtraction
paired every station with itself, and each station's range for the year is between 25.4 and 25.6 degrees,
the formula's seasonal swing of 9 on either side of the mean, its daily swing of 3, and a little wobble.

### A label missing from one side

Every station in `LATITUDES` should have sent a reading for each of the year's 8,760 hours. `count` gives
the readings each station actually sent, and the difference is the hours that are missing:

In [10]:
expected = pd.Series(8760, index=list(LATITUDES))
sent = readings.groupby("station")["celsius"].count()

print(expected - sent)

Bergen       0.0
Kirkenes     NaN
Oslo         0.0
Svalbard    24.0
Tromso       0.0
dtype: float64


`pd.Series(8760, index=...)` repeats one value for every label. Svalbard is 24 hours short, its silent
2 March. Kirkenes is in `expected` and not in `sent`, so its difference is `NaN`, and `NaN` turned the
whole Series into decimals, `float64`. When a missing label should count as zero, the `sub` method takes
a `fill_value`:

In [11]:
print(expected.sub(sent, fill_value=0))

Bergen         0.0
Kirkenes    8760.0
Oslo           0.0
Svalbard      24.0
Tromso         0.0
dtype: float64


Kirkenes sent nothing, so all 8,760 of its hours are missing. The result is still `float64`, because
pandas lines the labels up, with `NaN`, before it fills them.

### A Series indexed by time

A Series can be built from values and an index together. Indexed by the time of each reading, Oslo's
values can be looked up by date, and they line up with another station hour by hour:

In [12]:
oslo_by_time = pd.Series(oslo.to_numpy(), index=readings.loc[readings["station"] == "Oslo", "time"].to_numpy(),
                         name="Oslo")
bergen = readings[readings["station"] == "Bergen"]
bergen_by_time = pd.Series(bergen["celsius"].to_numpy(), index=bergen["time"].to_numpy(), name="Bergen")

print(oslo_by_time.head(3))
print("Oslo at 15:00 on 17 July:", oslo_by_time.loc["2025-07-17 15:00"])
print("Bergen minus Oslo, mean over the year:", round((bergen_by_time - oslo_by_time).mean(), 2))

2025-01-01 00:00:00   -3.5
2025-01-01 01:00:00   -5.3
2025-01-01 02:00:00   -5.3
Name: Oslo, dtype: float64
Oslo at 15:00 on 17 July: 19.3
Bergen minus Oslo, mean over the year: 1.5


`.to_numpy()` on both the values and the times drops the table's row labels, so the new Series has only
the times. `loc["2025-07-17 15:00"]` found Oslo's warmest hour of the year from text, because pandas reads
a date written that way. The two stations now share their labels, one for each hour, so the subtraction
compared the same hours, and Bergen was 1.5 degrees warmer on average, the difference between the
formula's two means. **The Index** notebook gets the same Series in one line with `set_index`.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/02-series-solutions.ipynb).

**1.** Make a Series from the dictionary `LATITUDES`, and print the name of the northernmost station with `idxmax`.

In [13]:
# your code here

**2.** Convert `planned` to Fahrenheit, and print only the stations whose mean is above 40 °F.

In [14]:
# your code here

**3.** Make Svalbard's readings a Series indexed by time, the way the last worked example made Oslo's, and print its reading at 15:00 on 13 July.

In [15]:
# your code here

**4.** Print the labels of Svalbard's first three readings in `readings["celsius"]` after filtering on the station, and say why they are not 0, 1 and 2.

In [16]:
# your code here

**5.** Index Tromso's and Oslo's readings by time, subtract Oslo's from Tromso's hour by hour, and print the mean of the difference rounded to two places.

In [17]:
# your code here

## Common errors

### No error, and every difference is NaN

In [18]:
bergen_celsius = readings[readings["station"] == "Bergen"]["celsius"]
difference = bergen_celsius - oslo

print(difference.head(4))
print(len(difference), "values,", difference.isna().sum(), "of them NaN")

0   NaN
1   NaN
4   NaN
5   NaN
Name: celsius, dtype: float64
17520 values, 17520 of them NaN


Both Series came from filtering the same table, so each kept its rows' labels: Bergen's are 0, 4, 8 and
onward, and Oslo's are 1, 5, 9. No label is in both, so pandas paired every value with nothing, and the
result has 17,520 labels, all of them `NaN`. Give both Series the same labels before combining them,
either the time of each reading, as the worked example did, or new positions from `reset_index(drop=True)`,
which **The Index** notebook explains:

In [19]:
fixed = bergen_celsius.reset_index(drop=True) - oslo.reset_index(drop=True)

print(fixed.head(3))
print("mean:", round(fixed.mean(), 2))

0   -0.1
1    1.6
2    1.6
Name: celsius, dtype: float64
mean: 1.5


The fix with positions works because both stations have a reading for every hour, in the same order. The
time index is the safer of the two, because it pairs readings by their times.

### FutureWarning: Series.__getitem__ treating keys as positions is deprecated

In [20]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    first = planned[0]

print(first)
for warning in caught:
    print(f"{warning.category.__name__}: {str(warning.message).split('. ')[0]}")

8.0


`planned` has text labels, so `planned[0]` cannot be a label, and pandas 2.2 treated the 0 as a position,
returned Bergen's 8.0 and warned. The rest of the message says that integer keys will always be treated as
labels in a future version, so the same line will then look for a label 0, which `planned` does not have,
as the next error shows with `loc`. Write `planned.iloc[0]` for a position, or `planned["Bergen"]` for a
label.

### KeyError: 0

In [21]:
planned.loc[0]

KeyError: 0

`loc` only ever looks up labels, and `planned` has no label 0, so `loc` raises without guessing. That makes
`loc` and `iloc` the two forms to prefer, since each one means only one thing.

### ValueError: Can only compare identically-labeled Series objects

In [22]:
highest > lowest

ValueError: Can only compare identically-labeled Series objects

Arithmetic lines two Series up by label, and a comparison operator does not. `highest` runs in
alphabetical order and `lowest` from coldest to warmest, so the two do not have the same labels in the
same order, and `>` raised. `gt`, the method form of `>`, lines the labels up first, as subtraction did:

In [23]:
print(highest.gt(lowest))

station
Bergen      True
Oslo        True
Svalbard    True
Tromso      True
Name: celsius, dtype: bool


Every station's highest reading is above its lowest. `lt`, `ge`, `le`, `eq` and `ne` are the other
comparisons that line up first.

## Recap

- Build a Series from a list, which numbers its values from 0, or from a dictionary, whose keys become the
  index
- Read a value by label with `loc` and by position with `iloc`, and tell when the two disagree
- Expect a filtered column to keep the labels its rows had in the table
- Predict the result of arithmetic between two Series from their labels, with `NaN` for a label only one
  side has, use `fill_value` when a missing label should count as zero, and compare with `gt` and its
  relatives when the labels are in different orders
- Index a Series by time so that two stations line up hour by hour

## What is next

The **DataFrame** notebook builds whole tables, from dictionaries, from rows and from a NumPy array, and
reads their shape and types before trusting them.

---

&#8592; **Previous:** [Why Pandas](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/01-why-pandas.ipynb)  &nbsp;·&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [DataFrame](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/03-dataframe.ipynb) &#8594;
